In [7]:
import pandas as pd
import numpy as np
from understatapi import UnderstatClient

In [8]:
TEAM_NAME = "Tottenham"
SEASONS = ["2022", "2023", "2024", "2025"]

In [76]:
all_matches = []

with UnderstatClient() as understat:
    for season in SEASONS:

        try:
            context_data = understat.team(team=TEAM_NAME).get_context_data(season=season)
            
            timing_dict = context_data.get("timing", {})

            # 順番を制御する＆扱いやすくするため
            interval_mapping = {
                "1-15": 1,
                "16-30": 2,
                "31-45": 3,
                "46-60": 4,
                "61-75": 5,
                "76+": 6
            }
            
            
            for time_interval, stats in timing_dict.items():
                row= {
                    "season": season,
                    "team": TEAM_NAME,
                    "interval": time_interval,
                    "interva_order": interval_mapping.get(time_interval, 99),
                    "shots": int(stats.get("shots", 0)),
                    "goals": int(stats.get("goals", 0)),
                    "xG": float(stats.get("xG", 0.0)),
                    "shots_against": int(stats.get("against", {}).get("shots", 0)),
                    "goals_against": int(stats.get("against", {}).get("goals", 0)),
                    "xGA": float(stats.get("against", {}).get("xG", 0.0))
                    
                }
                all_matches.append(row)

        except Exception as e:
            print(f"Error fetching data for season {season}: {e}")

df_raw = pd.DataFrame(all_matches)

In [77]:
df_raw.groupby(["season", "interval"])[["shots", "goals", "xG", "shots_against", "goals_against", "xGA"]].sum()

shots  goals         xG  shots_against  goals_against  \
season interval                                                          
2022   1-15         69      8   7.638104             81             14   
       16-30        73      7   7.365678             82              8   
       31-45        66      9   7.997466             90              9   
       46-60       117     13  14.285550            103             11   
       61-75        91     17  11.876871             82              9   
       76+         106     16  11.273399             82             12   
2023   1-15         76     12  10.713349             88              8   
       16-30        84      5   9.597448             72             12   
       31-45        72      8   8.765415             84              8   
       46-60       128     20  17.975499             91             12   
       61-75       103     10  11.646524             46              8   
       76+         124     19  20.059261            109             13   
2024   1-15         89     15  12.632541             85              9   
       16-30        78     11  10.057902             78             10   
       31-45        71      5   9.221625             83             14   
       46-60        92     11  14.296009            105             12   
       61-75        73      5   6.921282             85             10   
       76+          96     17  15.981305             90             10   
2025   1-15         60      4   4.582189             59              7   
       16-30        47      5   5.818575             67              5   
       31-45        76      9   9.471315             69             16   
       46-60        90     10   9.761025             98              9   
       61-75        60      7   6.861566             76              8   
       76+         103     13  12.440252             96             12   

                       xGA  
season interval             
2022   1-15      10.373608  
       16-30      5.699483  
       31-45      9.709004  
       46-60     10.835323  
       61-75      7.459653  
       76+        8.949331  
2023   1-15      14.168164  
       16-30     11.445592  
       31-45     10.864183  
       46-60     14.519957  
       61-75      6.742429  
       76+       14.456508  
2024   1-15      10.383783  
       16-30      9.556515  
       31-45     15.342355  
       46-60     16.547155  
       61-75     11.483925  
       76+        9.988457  
2025   1-15       6.420708  
       16-30      6.589866  
       31-45     10.298363  
       46-60     11.258525  
       61-75      8.928337  
       76+       12.630538

In [73]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 24 entries, 0 to 23
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   season         24 non-null     str    
 1   team           24 non-null     str    
 2   interval       24 non-null     str    
 3   shots          24 non-null     int64  
 4   goals          24 non-null     int64  
 5   xG             24 non-null     float64
 6   shots_against  24 non-null     int64  
 7   goals_against  24 non-null     int64  
 8   xGA            24 non-null     float64
dtypes: float64(2), int64(4), str(3)
memory usage: 1.8 KB


In [75]:
df_raw.groupby("interval")[["shots", "goals", "xG", "shots_against", "goals_against", "xGA"]].sum().reset_index()

,interval,shots,goals,xG,shots_against,goals_against,xGA
0,1-15,294,39,35.566183,313,38,41.346262
1,16-30,282,28,32.839603,299,35,33.291456
2,31-45,285,31,35.455821,326,47,46.213904
3,46-60,427,54,56.318084,397,44,53.160959
4,61-75,327,39,37.306242,289,35,34.614343
5,76+,429,65,59.754219,377,47,46.024835
